# Giano reconstruction example

See what Giano reconstructs when known observations are temporarily hidden in a random 72-hour validation window.

**Inputs:** `data/2-processed-v2/` and the corrected seed-42 weights in `checkpoints/corrected_v2_spectral/imputeformer/`.
Change `VARIABLE` below to use another variable's weights. The model is already trained.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from giano.evaluation.gapfill.reconstruction_example import main as export_example
from giano.meteorology import UNITS_BY_VARIABLE

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
VARIABLE = "temperature"
artifact_path = (
    project_root / f"artifacts/examples/corrected_v2_release/{VARIABLE}_random.json"
)
export_example(
    [
        "--data-dir",
        str(project_root / "data/2-processed-v2"),
        "--checkpoint-dir",
        str(project_root / "checkpoints/corrected_v2_spectral/imputeformer"),
        "--variable",
        VARIABLE,
        "--output",
        str(artifact_path),
    ]
)
example = json.loads(artifact_path.read_text(encoding="utf-8"))
series = pd.DataFrame(example["series"])
series["timestamps"] = pd.to_datetime(series["timestamps"])
series.head()

## Selected window and hidden-point error

The table identifies the sampled window and compares Giano with interpolation on the same hidden values. Lower MAE is better.


In [ ]:
pd.DataFrame(
    {
        "value": [
            example["variable"],
            example["station"],
            example["split"],
            example["sample_index"],
            example["seed"],
            example["case"]["mask_type"],
            example["case"]["parameter"],
            example["metrics"]["hidden_points"],
            example["metrics"]["model_mae"],
            example["metrics"]["interpolation_mae"],
        ]
    },
    index=[
        "variable",
        "station",
        "split",
        "window index",
        "training/mask seed",
        "mask",
        "parameter",
        "hidden points",
        "Giano MAE",
        "interpolation MAE",
    ],
)

In [ ]:
import matplotlib.dates as mdates

times = series["timestamps"]
mask = series["mask"].to_numpy(dtype=bool)
hidden_indices = np.flatnonzero(mask)
groups = np.split(hidden_indices, np.where(np.diff(hidden_indices) > 1)[0] + 1)

fig, ax = plt.subplots(figsize=(12, 5), dpi=130, layout="constrained")
for group in groups:
    if group.size:
        ax.axvspan(
            times.iloc[group[0]] - pd.Timedelta(minutes=30),
            times.iloc[group[-1]] + pd.Timedelta(minutes=30),
            color="#e7edf3",
            alpha=0.7,
            zorder=0,
        )
ax.plot(
    times,
    series["corrupted"],
    color="#777777",
    linewidth=1.4,
    marker="o",
    markersize=3,
    label="Visible observations",
)
ax.plot(
    times,
    series["ground_truth"].where(mask),
    color="#242424",
    linewidth=1.6,
    linestyle="--",
    marker="x",
    markersize=5,
    label="Hidden ground truth",
    zorder=4,
)
ax.plot(
    times,
    series["interpolation"].where(mask),
    color="#d78318",
    linewidth=1.8,
    linestyle="--",
    marker=".",
    markersize=4,
    label="Interpolation",
)
ax.plot(
    times,
    series["reconstruction"].where(mask),
    color="#1874d1",
    linewidth=2.2,
    marker="o",
    markersize=3.5,
    label="Giano reconstruction",
    zorder=3,
)

mask_type = example["case"]["mask_type"]
parameter = example["case"]["parameter"]
if mask_type == "point":
    mask_label = f"{parameter:.0%} randomly hidden points"
elif mask_type == "empirical":
    mask_label = f"Empirical gap · training quantile {parameter:.0%}"
else:
    mask_label = f"{parameter:g}-hour {mask_type.replace('_', ' ')} gap"
variable_label = example["variable"].replace("_", " ").title()
raw_unit = UNITS_BY_VARIABLE[example["variable"]]
unit = {"degC": "°C", "m s-1": "m/s", "degree": "°"}.get(raw_unit, raw_unit)
ax.set_title(f"{variable_label} · Station {example['station']} · {mask_label}", pad=14)
ax.set_ylabel(f"{variable_label} ({unit})")
ax.set_xlabel("Time")
locator = mdates.AutoDateLocator(minticks=3, maxticks=8)
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
ax.set_axisbelow(True)
ax.grid(axis="y", alpha=0.18)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.margins(x=0.015, y=0.12)
ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.16),
    ncol=4,
    frameon=False,
    fontsize=10,
)
plt.show()

## Reading the reconstruction

Grey points and lines are visible inputs. Shaded intervals contain artificially hidden observations: black crosses and dashes show their known truth, blue is Giano and orange is interpolation. Predictions are shown only at hidden points; natural missing values remain blank.

Rerun for another random window. To reproduce one, use its recorded `sample_index` with the export CLI's `--sample-index` and the same variable, seed and mask.
Use notebook 02 for the complete comparison and the [Wiki](https://github.com/itsminni/giano/wiki/Results-and-Evidence) for interpretation.


## Real gaps — before and after reconstruction

Temperature at Aldeno (San Zeno), 18–21 January 2023: two natural gaps of 3 and 14 hours. This section reads the precomputed Giano reconstructions from the website history export, independently of the random example above. No observations are artificially hidden. Ground truth, MAE and RMSE are unavailable for these missing measurements.

Run the cell below on its own. It shows the original and reconstructed series and saves a PNG in `artifacts/video/`.


In [ ]:
import json
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
history_path = (
    project_root
    / "artifacts/website/giano-history/histories/T0146/temperature/2023.json"
)
history = json.loads(history_path.read_text(encoding="utf-8"))
real_series = (
    pd.DataFrame(
        {"observed": history["observed"], "estimate": history["giano_estimates"]},
        index=pd.date_range(
            history["time_start"],
            periods=history["length"],
            freq=pd.Timedelta(seconds=history["step_seconds"]),
        ),
    )
    .loc["2023-01-18 12:00":"2023-01-21 11:00"]
    .astype(float)
)
missing = real_series["observed"].isna()
filled = real_series["observed"].fillna(real_series["estimate"])
edges = np.diff(np.r_[False, missing.to_numpy(), False].astype(int))
real_gaps = list(
    zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1), strict=True)
)
if len(real_series) != 72 or not missing.any():
    raise ValueError("Expected a 72-hour window containing natural gaps")
# Join estimates to the adjacent observations without filling unresolved gaps.
adjacent = missing.shift(1, fill_value=False) | missing.shift(-1, fill_value=False)
reconstructed = filled.where(missing | adjacent)

video_fig, video_axes = plt.subplots(
    2, 1, figsize=(12, 7), dpi=160, sharex=True, sharey=True, layout="constrained"
)
video_fig.suptitle("Aldeno (San Zeno) · Temperature · Real gaps", fontsize=16)
for video_ax, title in zip(
    video_axes, ["Original observations", "With Giano reconstruction"], strict=True
):
    for start, end in real_gaps:
        video_ax.axvspan(
            real_series.index[int(start)] - pd.Timedelta(minutes=30),
            real_series.index[int(end) - 1] + pd.Timedelta(minutes=30),
            color="#e7edf3",
            alpha=0.7,
            zorder=0,
        )
    video_ax.plot(
        real_series.index,
        real_series["observed"],
        color="#777777",
        linewidth=1.6,
        marker="o",
        markersize=3,
        label="Observed",
    )
    video_ax.set_title(title, loc="left", fontsize=11)
    video_ax.set_ylabel("Temperature (°C)")
    video_ax.grid(axis="y", alpha=0.18)
    video_ax.spines[["top", "right"]].set_visible(False)
    video_ax.margins(x=0.015, y=0.12)
video_axes[1].plot(
    real_series.index,
    reconstructed,
    color="#1874d1",
    linewidth=2.2,
    linestyle="--",
    label="Giano reconstruction",
)
video_locator = mdates.AutoDateLocator(minticks=4, maxticks=8)
video_axes[1].xaxis.set_major_locator(video_locator)
video_axes[1].xaxis.set_major_formatter(mdates.ConciseDateFormatter(video_locator))
video_axes[1].set_xlabel("Time")
video_axes[1].legend(
    loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2, frameon=False
)
video_path = project_root / "artifacts/video/04_temperature_real_gaps.png"
video_path.parent.mkdir(parents=True, exist_ok=True)
video_fig.savefig(video_path, dpi=160)
plt.show()
print(
    f"{int(missing.sum())} missing hours · {int((missing & filled.notna()).sum())} reconstructed"
)
print(video_path.relative_to(project_root))